In [1]:
import os
import shutil
import tempfile
from pathlib import Path
from uuid import uuid4

import ee
import geopandas as gpd
import sqlalchemy
from google.cloud import storage

In [2]:
ee.Initialize()

In [3]:
engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}",
)

In [4]:
ASSET_ID = "projects/your-project/assets/my_population_polygons"
GCS_BUCKET_NAME = "ee-temporary-bucket"
GCS_BLOB_NAME = "temp_ee_upload_polygons.zip"

In [14]:
def asset_exists(asset_id: str) -> bool:
    try:
        ee.data.getAsset(asset_id)
    except ee.ee_exception.EEException as e:
        if "not found" in str(e).lower():
            return False
        raise
    else:
        return True


def upload_to_gcs(local_path: os.PathLike, bucket_name: str, blob_name: str) -> str:
    print(f"Uploading {local_path} to gs://{bucket_name}/{blob_name}...")
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(local_path)
    print("Upload to GCS complete.")
    return f"gs://{bucket_name}/{blob_name}"


def delete_from_gcs(bucket_name: str, blob_name: str) -> None:
    print("Cleaning up temporary file in GCS...")
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.delete()
    print("Cleanup complete.")


def generate_and_upload_geometries(conn: sqlalchemy.Connection) -> str:
    df_agebs = gpd.read_postgis(
        """
        SELECT census_2020_ageb.cvegeo, census_2020_ageb.geometry
        FROM census_2020_ageb
        INNER JOIN census_2020_mun
            ON census_2020_ageb.cve_mun = census_2020_mun.cvegeo
        WHERE census_2020_mun.cve_met IS NOT NULL
        """,
        conn,
        geom_col="geometry",
    ).set_index("cvegeo")

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir_path = Path(tmpdir)
        df_agebs.to_file(tmpdir_path / "agebs.shp")

        with tempfile.TemporaryDirectory() as zip_tmpdir:
            zip_tmpdir_path = Path(zip_tmpdir)
            shutil.make_archive(str(zip_tmpdir_path / "compressed"), "zip", tmpdir_path)

            gcs_uri = upload_to_gcs(
                zip_tmpdir_path / "compressed.zip",
                GCS_BUCKET_NAME,
                GCS_BLOB_NAME,
            )
    return gcs_uri

In [ ]:
for year in [2020]:
    asset_id = f"agebs_{year}"

    if asset_exists(asset_id):
        print("Asset already exists")
        continue
    print("Asset not found")

    with engine.connect() as conn:
        gcs_uri = generate_and_upload_geometries(conn)

    print("Triggering Earth Engine table ingestion...")
    request_id = str(uuid4().hex)

    ingestion_params = {
        "id": asset_id,
        "sources": [{"uris": [gcs_uri]}],
    }

    try:
        task = ee.data.startTableIngestion(request_id, ingestion_params)
        task_id = task["id"]
        print(f"Task submitted successfully. Task ID: {task_id}")
    except Exception as e:
        print(f"Failed to start ingestion task: {e}")
        delete_from_gcs(GCS_BUCKET_NAME, GCS_BLOB_NAME)

Asset not found
Uploading C:\Users\lain\AppData\Local\Temp\tmpvq30y2xo\compressed.zip to gs://ee-temporary-bucket/temp_ee_upload_polygons.zip...
Upload to GCS complete.
